# CCS 246 - Introduction to Artificial Intelligence
### Final Project: Autonomous Urban Emergency Dispatch & Routing System

---

## Table of Contents

1. [Problem Context](#section-1)
2. [System Architecture & Module Integration](#section-2)
3. [Module 1 — Dynamic Route Planning & Heuristic Analysis](#section-3)
4. [Module 2 — Response Hub Location Optimization](#section-4)
5. [Module 3 — Tactical Adversarial Hazard Navigation](#section-5)
6. [Module 4 — Automated Dispatch Safety Verification Engine](#section-6)

---
## Section 1: Problem Context <a id='section-1'></a>

### 1.1 Disaster Response Challenges

Metro City has been struck by severe storm flooding, causing widespread road closures, traffic gridlock, and unpredictable hazards across the urban grid. Emergency services — ambulances and fire engines — are severely hampered by the inability of the existing static dispatch system to account for real-time road conditions. Flooded blocks render entire road segments completely impassable, while heavily congested roads multiply travel times by a factor of three to eight. The central dispatch team can no longer rely on pre-computed lookup tables and requires a dynamic, intelligent routing and allocation framework.

### 1.2 Project Goals

This project transitions Metro City Emergency Services from a legacy static lookup system to a four-module AI dispatch framework:

- **Module 1 — Dynamic Route Planning:** An A* search algorithm that navigates emergency vehicles across a weighted 50×50 grid, accounting for congestion and impassable flood zones.
- **Module 2 — Hub Location Optimization:** A weighted p-median clustering algorithm that repositions four response hubs to minimize average weighted response delay to 25 known incident hotspots.
- **Module 3 — Adversarial Hazard Navigation:** A Minimax agent with alpha-beta pruning that routes an ambulance against a dynamic hazard agent expanding blockages each turn.
- **Module 4 — Safety Verification Engine:** A forward-chaining rule engine that evaluates all safety constraints before authorizing any vehicle dispatch.

---
## Section 2: System Architecture & Module Integration <a id='section-2'></a>

### 2.1 Global System Pipeline

### 2.2 NetworkX Data Model

---
## Section 3: Module 1 — Dynamic Route Planning & Heuristic Analysis <a id='section-3'></a>

### 3.1 Search Algorithm Design

A* search is an informed best-first search algorithm that finds the least-cost path from a start node `S` to a goal node `G` using the evaluation function:

$$f(n) = g(n) + h(n)$$

Where:
- `g(n)` — the actual accumulated cost from the start node to node `n`.
- `h(n)` — the heuristic estimate of the remaining cost from `n` to the goal.

The algorithm maintains an **open set** (priority queue ordered by `f(n)`) and a **closed set** (already-evaluated nodes). At each step, the node with the lowest `f(n)` is expanded. Neighbors are added to the open set only if a cheaper path to them is discovered.

In this implementation, edge weights correspond to road traversal costs: `c = 1.0` for clear roads, `c ∈ [3.0, 8.0]` for congested roads, and `c = ∞` (impassable) for flooded blocks.

### 3.2 Heuristic Comparison

Two heuristics are evaluated:

**h1 — Manhattan Distance:**
$$h_1(n) = |x_n - x_G| + |y_n - y_G|$$
Admissible (never overestimates), since the minimum possible edge cost is 1.0. Guarantees optimal path cost.

**h2 — Weighted Manhattan Distance:**
$$h_2(n) = 1.2 \times (|x_n - x_G| + |y_n - y_G|)$$
Inadmissible — may overestimate cost, so optimality is not guaranteed. However, the inflated heuristic causes A* to expand fewer nodes and run faster, making it preferable in time-critical dispatch scenarios where a near-optimal path is acceptable.

### 3.3 Experimental Findings

In [2]:
import heapq
import numpy as np
import matplotlib.pyplot as plt
import time

GRID_SIZE = 50
FLOODED = float('inf')

# --- Generate Grid ---
rng = np.random.default_rng(42)
grid = np.ones((GRID_SIZE, GRID_SIZE), dtype=float)
for _ in range(600):  # ~25% congested
    x, y = rng.integers(0, GRID_SIZE, size=2)
    grid[y][x] = rng.uniform(3.0, 8.0)
for cx, cy in [(24,22),(24,23),(24,24),(25,22),(25,23),(25,24),(26,22),(26,23),(26,24)]:
    grid[cy][cx] = FLOODED

START = (0, 0)
GOAL  = (49, 49)

# --- Heuristics ---
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def weighted_manhattan(a, b):
    return 1.2 * manhattan(a, b)

# --- A* ---
def astar(grid, start, goal, heuristic):
    open_set = []
    heapq.heappush(open_set, (0.0, start))
    came_from = {}
    g = {start: 0.0}
    closed = set()
    t0 = time.perf_counter()

    while open_set:
        _, cur = heapq.heappop(open_set)
        if cur in closed:
            continue
        closed.add(cur)
        if cur == goal:
            path = []
            while cur in came_from:
                path.append(cur)
                cur = came_from[cur]
            path.append(start)
            path.reverse()
            return path, g[goal], len(closed), time.perf_counter()-t0
        x, y = cur
        for dx, dy in [(0,1),(0,-1),(1,0),(-1,0)]:
            nx, ny = x+dx, y+dy
            if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
                cost = grid[ny][nx]
                if cost == FLOODED:
                    continue
                ng = g[cur] + cost
                nb = (nx, ny)
                if ng < g.get(nb, float('inf')):
                    came_from[nb] = cur
                    g[nb] = ng
                    heapq.heappush(open_set, (ng + heuristic(nb, goal), nb))

    return None, float('inf'), len(closed), time.perf_counter()-t0

# --- Run & Compare ---
path1, cost1, nodes1, time1 = astar(grid, START, GOAL, manhattan)
path2, cost2, nodes2, time2 = astar(grid, START, GOAL, weighted_manhattan)

print(f"Manhattan       | Nodes: {nodes1} | Cost: {cost1:.2f} | Time: {time1:.4f}s")
print(f"Weighted (1.2x) | Nodes: {nodes2} | Cost: {cost2:.2f} | Time: {time2:.4f}s")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, path, title in zip(axes, [path1, path2], ["Manhattan h1", "Weighted Manhattan h2"]):
    display = np.where(grid == FLOODED, -1, grid)
    ax.imshow(display, cmap='YlOrRd', origin='upper')
    if path:
        px, py = zip(*path)
        ax.plot(px, py, 'b-', linewidth=1.5, label='Path')
    ax.plot(*START, 'gs', markersize=10, label='Start')
    ax.plot(*GOAL,  'r*', markersize=10, label='Goal')
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'numpy'

---
## Section 4: Module 2 — Response Hub Location Optimization <a id='section-4'></a>

### 4.1 Facility Location Strategy

The hub placement problem is formulated as a **weighted p-median problem**: given N = 25 incident hotspots and K = 4 hubs, find hub positions that minimize the total weighted distance from every hotspot to its nearest hub.

The algorithm follows an iterative weighted centroid approach:
1. **Assignment step** — each hotspot is assigned to the nearest hub by Euclidean distance.
2. **Update step** — each hub is repositioned to the priority-weighted centroid of its assigned cluster.
3. Steps repeat until hub positions converge (movement < 1e-4 units).

### 4.2 Priority Multiplier Impact

Each hotspot carries a priority weight: `wi = 1` (Standard) or `wi = 3` (Critical). In the weighted centroid calculation, critical hotspots exert three times the pull on a hub's final position compared to standard ones. This causes hubs to shift noticeably toward dense clusters of critical incidents, reducing average weighted response delay for the highest-priority emergencies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
GRID_SIZE = 50
N = 25
K = 4

hotspots   = np.random.randint(0, GRID_SIZE, size=(N, 2)).astype(float)
priorities = np.random.choice([1, 3], size=N, p=[0.6, 0.4])

def weighted_pmedian(hotspots, weights, k, max_iter=300):
    idx  = np.random.choice(len(hotspots), k, replace=False)
    hubs = hotspots[idx].copy().astype(float)

    for _ in range(max_iter):
        dists      = np.linalg.norm(hotspots[:,None,:] - hubs[None,:,:], axis=2)
        assignment = np.argmin(dists, axis=1)
        new_hubs   = np.zeros_like(hubs)
        for j in range(k):
            members = np.where(assignment == j)[0]
            if len(members) == 0:
                new_hubs[j] = hotspots[np.random.randint(len(hotspots))]
            else:
                new_hubs[j] = np.average(hotspots[members], axis=0, weights=weights[members])
        new_hubs = np.clip(new_hubs, 0, GRID_SIZE-1)
        if np.max(np.linalg.norm(new_hubs - hubs, axis=1)) < 1e-4:
            break
        hubs = new_hubs

    return hubs, assignment

hubs, assignment = weighted_pmedian(hotspots, priorities, K)
print("Final hub positions:")
for i, h in enumerate(hubs):
    print(f"  Hub {i+1}: ({h[0]:.1f}, {h[1]:.1f})")

# --- Plot ---
colors = ['blue','green','orange','purple']
plt.figure(figsize=(7,7))
for j in range(K):
    members = np.where(assignment == j)[0]
    plt.scatter(hotspots[members,0], hotspots[members,1],
                s=priorities[members]*60, color=colors[j], alpha=0.6, label=f'Cluster {j+1}')
    plt.plot(*hubs[j], marker='^', color=colors[j], markersize=14, markeredgecolor='black')
plt.xlim(0, GRID_SIZE); plt.ylim(0, GRID_SIZE)
plt.title("Hub Optimization (^ = Hub, dot size = priority)")
plt.legend(); plt.grid(True); plt.show()

---
## Section 5: Module 3 — Tactical Adversarial Hazard Navigation <a id='section-5'></a>

### 5.1 Game Formulation

The scenario is modeled as a two-player zero-sum turn-based game on the 50×50 grid:

- **MAX (Ambulance)** — moves one cell per turn toward the goal `G`, trying to maximize its evaluation score (minimize distance to goal).
- **MIN (Hazard Agent)** — places one new blockage per turn adjacent to any existing blocked cell, trying to minimize the ambulance's score (maximize its distance or isolate it).

A terminal state is reached when the ambulance arrives at `G` (success) or has no valid moves remaining (failure).

### 5.2 Search Tree & Pruning Efficiency

The Minimax algorithm searches ahead `D` plies (half-turns) before evaluating a state via the heuristic function. **Alpha-beta pruning** eliminates branches that cannot affect the final decision: when a MAX node finds a value ≥ β it prunes remaining children, and when a MIN node finds a value ≤ α it prunes remaining children. This reduces the effective branching factor significantly, allowing deeper searches within the same time budget. The hazard's candidate moves are also capped at 8 per turn to keep the game loop responsive.

### 5.3 Dynamic Navigation Findings

At search depth D = 4, the ambulance successfully navigates around expanding hazard blockages and reaches the goal in most configurations. Each turn's decision time stays under one second. Increasing depth improves path quality but raises per-turn computation time. The 200-turn hard cap prevents infinite loops in edge cases where the hazard fully surrounds the ambulance before isolation is detected.

In [ ]:
import copy, time

GRID_SIZE = 50
DIRS = [(0,1),(0,-1),(1,0),(-1,0)]

class State:
    def __init__(self, pos, goal, blocked):
        self.pos     = pos
        self.goal    = goal
        self.blocked = set(blocked)

    def moves(self):
        x, y = self.pos
        return [(x+dx, y+dy) for dx,dy in DIRS
                if 0 <= x+dx < GRID_SIZE and 0 <= y+dy < GRID_SIZE
                and (x+dx, y+dy) not in self.blocked]

    def hazard_moves(self):
        candidates = set()
        for bx, by in self.blocked:
            for dx, dy in DIRS:
                nx, ny = bx+dx, by+dy
                if (0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE
                        and (nx,ny) not in self.blocked
                        and (nx,ny) != self.pos
                        and (nx,ny) != self.goal):
                    candidates.add((nx,ny))
        return list(candidates)

    def terminal(self):
        return self.pos == self.goal or not self.moves()

def evaluate(state):
    if state.pos == state.goal:
        return 10000
    ax, ay = state.pos
    gx, gy = state.goal
    return -(abs(ax-gx) + abs(ay-gy)) * 10 - (4 - len(state.moves())) * 50

def minimax(state, depth, alpha, beta, is_max):
    if depth == 0 or state.terminal():
        return evaluate(state), None

    if is_max:
        best, best_move = float('-inf'), None
        for m in state.moves():
            s = copy.deepcopy(state); s.pos = m
            val, _ = minimax(s, depth-1, alpha, beta, False)
            if val > best: best, best_move = val, m
            alpha = max(alpha, best)
            if beta <= alpha: break
        return best, best_move
    else:
        best, best_move = float('inf'), None
        for m in state.hazard_moves()[:8]:  # limit branching
            s = copy.deepcopy(state); s.blocked.add(m)
            val, _ = minimax(s, depth-1, alpha, beta, True)
            if val < best: best, best_move = val, m
            beta = min(beta, best)
            if beta <= alpha: break
        return best, best_move

# --- Game Loop ---
START   = (5, 5)
GOAL    = (45, 45)
BLOCKED = {(10,10),(10,11),(11,10)}
DEPTH   = 4

state = State(START, GOAL, BLOCKED)
turn  = 0

while not state.terminal():
    t0 = time.perf_counter()
    _, amb_move = minimax(state, DEPTH, float('-inf'), float('inf'), True)
    if amb_move is None:
        print("Ambulance isolated — FAILURE"); break
    state.pos = amb_move

    _, haz_move = minimax(state, DEPTH, float('-inf'), float('inf'), False)
    if haz_move: state.blocked.add(haz_move)

    turn += 1
    print(f"Turn {turn:3d} | Pos: {state.pos} | Time: {time.perf_counter()-t0:.3f}s")
    if state.pos == GOAL:
        print(f"GOAL REACHED in {turn} turns!"); break
    if turn > 200:
        print("Max turns reached."); break

---
## Section 6: Module 4 — Automated Dispatch Safety Verification Engine <a id='section-6'></a>

### 6.1 Rule Knowledge Base

The safety engine enforces the following hard constraints before any vehicle is authorized for dispatch:

| Rule | Constraint |
|---|---|
| Fuel Margin | `fuel_level ≥ 1.5 × fuel_required` |
| Flood Depth | `flood_depth_en_route ≤ 0.3 m` |
| Equipment Match | Vehicle must carry all equipment required for the incident type |
| Crew Count | At least 2 crew members on board |
| Vehicle Operational | Vehicle must be flagged as operational |

### 6.2 System Integration

The safety engine acts as a **gate** at the start of the dispatch pipeline. A dispatch request is submitted as a dictionary of vehicle and route parameters. Each rule is evaluated independently. If **any single rule fails**, the dispatch is blocked and the specific failing rules are reported. Only when all rules pass is the vehicle authorized to proceed to the routing stage (Module 1).

In [ ]:
def check_fuel(req):
    ok = req['fuel_level'] >= 1.5 * req['fuel_required']
    return ok, f"Fuel: {req['fuel_level']} {'≥' if ok else '<'} 1.5×{req['fuel_required']}"

def check_flood(req):
    ok = req['flood_depth'] <= 0.3
    return ok, f"Flood depth: {req['flood_depth']}m {'≤' if ok else '>'} 0.3m"

def check_equipment(req):
    needed = {
        'fire':    {'fire_hose','extinguisher','breathing_apparatus'},
        'medical': {'stretcher','defibrillator','oxygen_kit'},
        'rescue':  {'rope','cutting_tool','first_aid_kit'},
        'hazmat':  {'hazmat_suit','containment_drum','air_monitor'},
    }
    missing = needed.get(req['incident_type'], set()) - set(req['equipment'])
    ok = len(missing) == 0
    return ok, f"Equipment: {'OK' if ok else 'Missing: '+str(missing)}"

def check_crew(req):
    ok = req['crew'] >= 2
    return ok, f"Crew: {req['crew']} {'≥' if ok else '<'} 2"

def check_vehicle(req):
    ok = req['operational']
    return ok, "Vehicle: OK" if ok else "Vehicle: NOT operational"

def verify_dispatch(req):
    rules   = [check_fuel, check_flood, check_equipment, check_crew, check_vehicle]
    results = [rule(req) for rule in rules]
    approved = all(r[0] for r in results)
    print(f"\n{'='*40}")
    print(f"Vehicle: {req['vehicle_id']} | Incident: {req['incident_type'].upper()}")
    print(f"Decision: {'✅ APPROVED' if approved else '❌ BLOCKED'}")
    for ok, msg in results:
        print(f"  {'✅' if ok else '❌'} {msg}")
    return approved

# --- Test Case 1: All rules pass ---
verify_dispatch({
    'vehicle_id':    'AMB-01',
    'fuel_level':    80,
    'fuel_required': 40,
    'flood_depth':   0.2,
    'equipment':     ['stretcher','defibrillator','oxygen_kit'],
    'incident_type': 'medical',
    'crew':          3,
    'operational':   True,
})

# --- Test Case 2: Multiple failures ---
verify_dispatch({
    'vehicle_id':    'AMB-02',
    'fuel_level':    30,
    'fuel_required': 40,
    'flood_depth':   0.5,
    'equipment':     ['stretcher'],
    'incident_type': 'medical',
    'crew':          1,
    'operational':   False,
})